# YUVAA Internship — Week 1
## Exploratory Data Analysis of Crop Production in India

**Prepared by:** Avirup Karmakar

This notebook documents the Week 1 workflow: dataset loading, data-quality assessment, cleaning, descriptive statistics, exploratory analysis, visualization, and initial findings.

**Dataset:** `crop_production.csv`

**Columns:** `State_Name`, `District_Name`, `Crop_Year`, `Season`, `Crop`, `Area`, `Production`

> **Cleaning decision:** 3,730 missing values were identified in `Production`. They are excluded from production-based analysis rather than replaced with zero.


## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
print("Libraries imported successfully!")


## 2. Load Dataset

In [ ]:
df = pd.read_csv("crop_production.csv")

print("Dataset loaded successfully!")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])
display(df.head())


## 3. Initial Inspection

In [ ]:
print("Column names:")
print(df.columns.tolist())

print("\nData types:")
display(df.dtypes.to_frame("Data Type"))

print("\nDataset information:")
df.info()


In [ ]:
print("Descriptive overview:")
display(df.describe(include="all").T)


## 4. Missing-Value Analysis

In [ ]:
missing_summary = pd.DataFrame({
    "Missing Values": df.isnull().sum(),
    "Missing Percentage": (df.isnull().sum() / len(df) * 100).round(2)
})
display(missing_summary)


### Missing-value treatment

`Production` contains **3,730** missing values. Because production is a key variable for the analysis, these rows are removed from production-based calculations. Missing production is not assumed to mean zero production.


## 5. Duplicate and Validity Checks

In [ ]:
print("Duplicate rows:", int(df.duplicated().sum()))
print("Negative Area values:", int((pd.to_numeric(df["Area"], errors="coerce") < 0).sum()))
print("Negative Production values:", int((pd.to_numeric(df["Production"], errors="coerce") < 0).sum()))


## 6. Data Cleaning and Preprocessing

In [ ]:
df.columns = df.columns.str.strip()

for column in ["State_Name", "District_Name", "Season", "Crop"]:
    df[column] = df[column].astype("string").str.strip()

df["Crop_Year"] = pd.to_numeric(df["Crop_Year"], errors="coerce")
df["Area"] = pd.to_numeric(df["Area"], errors="coerce")
df["Production"] = pd.to_numeric(df["Production"], errors="coerce")

# Remove duplicate rows
df = df.drop_duplicates()

# Keep cleaned dataset
cleaned_df = df.copy()

# Exclude missing/invalid values only for analytical calculations
analysis_df = cleaned_df.dropna(
    subset=["Crop_Year", "Area", "Production"]
).copy()

analysis_df = analysis_df[
    (analysis_df["Area"] >= 0) &
    (analysis_df["Production"] >= 0)
].copy()

print("Cleaned dataset shape:", cleaned_df.shape)
print("Analysis dataset shape:", analysis_df.shape)


In [ ]:
print("Missing values after cleaning:")
display(cleaned_df.isnull().sum())

print("Duplicate rows after cleaning:", int(cleaned_df.duplicated().sum()))
display(cleaned_df.dtypes.to_frame("Data Type"))


In [ ]:
cleaned_df.to_csv("crop_production_cleaned.csv", index=False)
print("Saved: crop_production_cleaned.csv")


## 7. Dataset Profile

In [ ]:
print("Number of states/regions:", analysis_df["State_Name"].nunique())
print("Number of districts:", analysis_df["District_Name"].nunique())
print("Number of crops:", analysis_df["Crop"].nunique())
print("Number of seasons:", analysis_df["Season"].nunique())
print("Year range:", int(analysis_df["Crop_Year"].min()), "-", int(analysis_df["Crop_Year"].max()))

print("\nSeason counts:")
display(analysis_df["Season"].value_counts())


## 8. Descriptive Statistics

In [ ]:
display(
    analysis_df[["Crop_Year", "Area", "Production"]].describe()
)


## 9. Crop-wise Production Analysis

In [ ]:
crop_production = (
    analysis_df.groupby("Crop")["Production"]
    .sum()
    .sort_values(ascending=False)
)

display(crop_production.head(10).to_frame("Total Production"))


In [ ]:
top_crops = crop_production.head(10).sort_values()

plt.figure(figsize=(10, 6))
plt.barh(top_crops.index, top_crops.values)
plt.title("Top 10 Crops by Total Production")
plt.xlabel("Total Production")
plt.ylabel("Crop")
plt.tight_layout()
plt.savefig("top_10_crops.png", dpi=300)
plt.show()


## 10. State-wise Production Analysis

In [ ]:
state_production = (
    analysis_df.groupby("State_Name")["Production"]
    .sum()
    .sort_values(ascending=False)
)

display(state_production.head(10).to_frame("Total Production"))


In [ ]:
top_states = state_production.head(10).sort_values()

plt.figure(figsize=(10, 6))
plt.barh(top_states.index, top_states.values)
plt.title("Top 10 States by Total Crop Production")
plt.xlabel("Total Production")
plt.ylabel("State")
plt.tight_layout()
plt.savefig("top_10_states.png", dpi=300)
plt.show()


## 11. District-wise Production Analysis

In [ ]:
district_production = (
    analysis_df.groupby(["State_Name", "District_Name"])["Production"]
    .sum()
    .sort_values(ascending=False)
)

display(district_production.head(10).to_frame("Total Production"))


In [ ]:
top_districts = district_production.head(10).sort_values()
labels = [f"{district} ({state})" for state, district in top_districts.index]

plt.figure(figsize=(10, 7))
plt.barh(labels, top_districts.values)
plt.title("Top 10 Districts by Total Crop Production")
plt.xlabel("Total Production")
plt.ylabel("District (State)")
plt.tight_layout()
plt.savefig("top_10_districts.png", dpi=300)
plt.show()


## 12. Season-wise Production Analysis

In [ ]:
season_production = (
    analysis_df.groupby("Season")["Production"]
    .sum()
    .sort_values(ascending=False)
)

display(season_production.to_frame("Total Production"))


In [ ]:
plt.figure(figsize=(10, 6))
plt.bar(season_production.index, season_production.values)
plt.title("Total Crop Production by Agricultural Season")
plt.xlabel("Season")
plt.ylabel("Total Production")
plt.xticks(rotation=30)
plt.tight_layout()
plt.savefig("production_by_season.png", dpi=300)
plt.show()


## 13. Year-wise Production Analysis

In [ ]:
yearly_production = (
    analysis_df.groupby("Crop_Year")["Production"]
    .sum()
    .sort_index()
)

display(yearly_production.to_frame("Total Production"))


In [ ]:
plt.figure(figsize=(11, 6))
plt.plot(yearly_production.index, yearly_production.values, marker="o")
plt.title("Crop Production Trend by Year")
plt.xlabel("Year")
plt.ylabel("Total Production")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("production_trend.png", dpi=300)
plt.show()


## 14. Year-over-Year Growth

In [ ]:
yearly_growth = yearly_production.pct_change() * 100

growth_table = pd.DataFrame({
    "Year": yearly_growth.index,
    "Production Growth (%)": yearly_growth.values
})
display(growth_table)

valid_growth = yearly_growth.dropna()
if not valid_growth.empty:
    print("Highest YoY growth:", int(valid_growth.idxmax()), f"({valid_growth.max():.2f}%)")
    print("Largest YoY decline:", int(valid_growth.idxmin()), f"({valid_growth.min():.2f}%)")


## 15. Cultivated Area Analysis

In [ ]:
yearly_area = (
    analysis_df.groupby("Crop_Year")["Area"]
    .sum()
    .sort_index()
)

display(yearly_area.to_frame("Total Area"))


In [ ]:
plt.figure(figsize=(11, 6))
plt.plot(yearly_area.index, yearly_area.values, marker="o")
plt.title("Cultivated Area Trend by Year")
plt.xlabel("Year")
plt.ylabel("Total Area")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("area_trend.png", dpi=300)
plt.show()


## 16. Area vs Production

In [ ]:
sample_df = analysis_df.sample(
    n=min(15000, len(analysis_df)),
    random_state=42
)

plt.figure(figsize=(10, 6))
plt.scatter(sample_df["Area"], sample_df["Production"], alpha=0.35)
plt.title("Relationship Between Cultivated Area and Production")
plt.xlabel("Area")
plt.ylabel("Production")
plt.tight_layout()
plt.savefig("area_vs_production.png", dpi=300)
plt.show()


## 17. Correlation Analysis

In [ ]:
correlation_matrix = analysis_df[["Area", "Production"]].corr()
display(correlation_matrix)

area_production_corr = analysis_df["Area"].corr(
    analysis_df["Production"]
)
print(f"Area-Production correlation: {area_production_corr:.3f}")


## 18. Highest Production Record

In [ ]:
highest_record = analysis_df.loc[
    analysis_df["Production"].idxmax()
]
display(highest_record.to_frame("Value"))


## 19. Key Findings Summary

In [ ]:
print("Highest-production crop:", crop_production.index[0])
print("Highest-production state:", state_production.index[0])
print(
    "Highest-production district:",
    district_production.index[0][1],
    f"({district_production.index[0][0]})"
)
print("Highest-production season:", season_production.index[0])
print("Highest-production year:", int(yearly_production.idxmax()))
print("Area-Production correlation:", round(area_production_corr, 3))


## 20. Conclusion

Week 1 establishes a reproducible agricultural data-analysis workflow. The dataset is inspected, cleaned, summarized and explored across crop, state, district, season and year dimensions. Visual analysis is used to communicate trends and relationships. The resulting cleaned dataset can serve as the foundation for later agricultural analytics and predictive modelling.

**Note:** Interpretations in the final report should use the actual outputs generated by this notebook. Correlation should not be described as proof of causation.
